# Project: a whole behavioural experiment with pandas

Combine 16 tracking files with a metadata table, compare groups, follow the activity over time, and check the design.

The full explanations are in [experiment_pandas.md](experiment_pandas.md). *This work is licensed under [Creative Commons Attribution 4.0 International License](http://creativecommons.org/licenses/by/4.0/).*

## How to use this notebook

* Run the cells **in order**, with `Shift+Enter`.
* Cells starting with `%%writefile` **create a file** (module, tests or script)
  in the folder of this notebook: like in the lesson, the project is made of
  real `.py` files, that you can also open in a text editor or Spyder.
* `%run script.py` runs a script as from the shell, and the cell with
  `pytest` runs the tests.
* Start JupyterLab from an **empty folder** dedicated to this project, in the
  `sci` environment:

  ```
  (sci) $ mkdir my_project
  (sci) $ cd my_project
  (sci) $ jupyter lab
  ```

## Some data

16 mice, one tracking file each, and a metadata table (sex, treatment, cage).

In [ ]:
%%writefile generate_experiment.py
"""Simulate an open-field experiment: 16 mice, tracking files and metadata."""

from pathlib import Path

import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=4)
folder = Path("tracking")
folder.mkdir(exist_ok=True)


def simulate(speed, duration=600, fps=5, arena=50):
    """Random walk of a mouse in a square arena (positions in cm)."""
    n = duration * fps
    x, y = np.empty(n), np.empty(n)
    x[0] = y[0] = arena / 2
    angle = rng.uniform(0, 2 * np.pi)
    for i in range(1, n):
        habituation = 1.5 - i / n                  # mice slow down over time
        angle += rng.normal(0, 0.4)
        step = rng.exponential(speed * habituation / fps)
        x[i] = np.clip(x[i - 1] + step * np.cos(angle), 0, arena)
        y[i] = np.clip(y[i - 1] + step * np.sin(angle), 0, arena)
        if x[i] in (0, arena) or y[i] in (0, arena):
            angle += np.pi / 2 + rng.uniform(0, np.pi)
    return np.arange(n) / fps, x.round(2), y.round(2)


rows = []
for k in range(1, 17):
    mouse = f"M{k:02d}"
    sex = "F" if k % 2 else "M"
    treatment = "drug" if k > 8 else "vehicle"
    speed = (6 if treatment == "drug" else 9) + (1 if sex == "F" else 0)
    t, x, y = simulate(speed)
    pd.DataFrame({"time": t, "x": x, "y": y}).to_csv(folder / f"{mouse}.csv",
                                                      index=False)
    rows.append({"mouse": mouse, "sex": sex, "treatment": treatment,
                 "cage": f"C{(k - 1) // 4 + 1}"})

pd.DataFrame(rows).to_csv("metadata.csv", index=False)
print("Written 16 tracking files and metadata.csv")

In [ ]:
%run generate_experiment.py

## Step 1: one long table

`pd.concat` stacks the files into one long table. In `add_steps`,
`groupby("mouse")["x"].diff()` computes the differences **within each
mouse**.

In [ ]:
%%writefile behaviour.py
"""Combine and summarize tracking files with pandas."""

from pathlib import Path

import numpy as np
import pandas as pd


def load_tracking(folder):
    """Read all the tracking files of a folder into one long table.

    Each file is named after the mouse (e.g. M01.csv). The result has one row
    per frame, with the columns mouse, time, x and y.
    """
    tables = []
    for filename in sorted(Path(folder).glob("*.csv")):
        table = pd.read_csv(filename)
        table.insert(0, "mouse", filename.stem)
        tables.append(table)
    return pd.concat(tables, ignore_index=True)


def add_steps(tracking):
    """Add the column step: distance travelled since the previous frame (cm).

    The differences are computed within each mouse, so that the first frame
    of a mouse is not compared with the last frame of the previous one.
    """
    tracking = tracking.copy()
    dx = tracking.groupby("mouse")["x"].diff()
    dy = tracking.groupby("mouse")["y"].diff()
    tracking["step"] = np.hypot(dx, dy).fillna(0)
    return tracking


def summarize(tracking, arena=50, center=0.5):
    """Return one row per mouse: distance travelled (cm) and time in center (s)."""
    margin = arena * (1 - center) / 2
    tracking = tracking.assign(
        in_center=tracking["x"].between(margin, arena - margin)
        & tracking["y"].between(margin, arena - margin))
    dt = tracking["time"].iloc[1] - tracking["time"].iloc[0]
    summary = tracking.groupby("mouse").agg(distance=("step", "sum"),
                                            center_frames=("in_center", "sum"))
    summary["center_time"] = summary["center_frames"] * dt
    return summary.drop(columns="center_frames").round(1)

In [ ]:
import pandas as pd
from behaviour import load_tracking, add_steps, summarize

tracking = add_steps(load_tracking("tracking"))
print(tracking.shape)
summary = summarize(tracking)
summary.head(3)

## Step 2: tests

The test table contains two mice, to check that steps are not computed across mice.

In [ ]:
%%writefile test_behaviour.py
"""Tests for behaviour.py."""

import pandas as pd

from behaviour import add_steps, load_tracking, summarize


def two_mice():
    return pd.DataFrame({
        "mouse": ["A", "A", "A", "B", "B"],
        "time":  [0.0, 1.0, 2.0, 0.0, 1.0],
        "x":     [0.0, 3.0, 3.0, 25.0, 25.0],
        "y":     [0.0, 4.0, 14.0, 25.0, 25.0],
    })


def test_steps_are_computed_within_each_mouse():
    steps = add_steps(two_mice())["step"].tolist()
    assert steps == [0.0, 5.0, 10.0, 0.0, 0.0]


def test_summary_distance_and_center_time():
    summary = summarize(add_steps(two_mice()))
    assert summary.loc["A", "distance"] == 15.0
    assert summary.loc["A", "center_time"] == 0.0     # A stays near a corner
    assert summary.loc["B", "center_time"] == 2.0     # B: 2 frames of 1 s


def test_load_tracking(tmp_path):
    pd.DataFrame({"time": [0, 1], "x": [1, 2], "y": [3, 4]}).to_csv(
        tmp_path / "M01.csv", index=False)
    pd.DataFrame({"time": [0], "x": [5], "y": [6]}).to_csv(
        tmp_path / "M02.csv", index=False)
    tracking = load_tracking(tmp_path)
    assert tracking["mouse"].tolist() == ["M01", "M01", "M02"]
    assert list(tracking.columns) == ["mouse", "time", "x", "y"]

In [ ]:
import sys
!{sys.executable} -m pytest -q

## Step 3: adding the conditions

In [ ]:
metadata = pd.read_csv("metadata.csv")
results = summary.reset_index().merge(metadata, on="mouse")
results.head(3)

In [ ]:
results.groupby("treatment")["distance"].agg(["mean", "std", "count"]).round(1)

In [ ]:
results.pivot_table(values="distance", index="treatment", columns="sex").round(1)

## Step 4: the analysis script

In [ ]:
%%writefile analyze_experiment.py
"""Analyse a whole open-field experiment: tracking files + metadata."""

import argparse

import matplotlib.pyplot as plt
import pandas as pd

from behaviour import add_steps, load_tracking, summarize


def main():
    parser = argparse.ArgumentParser(description="Open-field experiment analysis.")
    parser.add_argument("folder", help="folder with one tracking CSV per mouse")
    parser.add_argument("metadata", help="CSV file: mouse, sex, treatment, ...")
    args = parser.parse_args()

    tracking = add_steps(load_tracking(args.folder))
    metadata = pd.read_csv(args.metadata)
    print(f"{tracking['mouse'].nunique()} mice, {len(tracking)} frames")

    results = summarize(tracking).reset_index().merge(metadata, on="mouse")
    results.to_csv("results.csv", index=False)
    print(results.groupby(["treatment", "sex"])[["distance", "center_time"]]
          .mean().round(1))

    tracking["minute"] = (tracking["time"] // 60).astype(int)
    per_minute = (tracking.groupby(["mouse", "minute"])["step"].sum()
                  .reset_index().merge(metadata, on="mouse"))
    activity = per_minute.groupby(["minute", "treatment"])["step"].mean().unstack()

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    activity.plot(ax=axes[0], marker="o")
    axes[0].set_xlabel("Minute")
    axes[0].set_ylabel("Distance per minute (cm)")
    axes[0].set_title("Activity over time")
    results.boxplot(column="distance", by=["treatment", "sex"], ax=axes[1])
    axes[1].set_ylabel("Total distance (cm)")
    axes[1].set_title("Distance travelled")
    fig.suptitle("")
    fig.tight_layout()
    fig.savefig("experiment.png", dpi=150)
    plt.show()


if __name__ == "__main__":
    main()

In [ ]:
%run analyze_experiment.py tracking metadata.csv

## Going further

**1.** Compute the mean distance per **cage**. What does it tell you about the design of the experiment?

<details><summary>Answer</summary>

```python
results.groupby("cage")["distance"].mean().round(0)
pd.crosstab(results["cage"], results["treatment"])
```

Each cage contains a single treatment: cage and treatment are **confounded**.

</details>

In [ ]:
# Your code here

**2.** Test the difference of distance between treatments with a Mann–Whitney test (`scipy.stats.mannwhitneyu`).

<details><summary>Answer</summary>

```python
from scipy import stats

drug = results.loc[results["treatment"] == "drug", "distance"]
vehicle = results.loc[results["treatment"] == "vehicle", "distance"]
stats.mannwhitneyu(drug, vehicle)
```

</details>

In [ ]:
# Your code here

**3.** Save the results in an Excel file with one sheet per treatment.

<details><summary>Answer</summary>

```python
with pd.ExcelWriter("results.xlsx") as writer:
    for treatment, group in results.groupby("treatment"):
        group.to_excel(writer, sheet_name=treatment, index=False)
```

</details>

In [ ]:
# Your code here